In [8]:
import warnings
from causalnex.structure import StructureModel
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [9]:
df_se = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/se_2007_2015_bn.csv")

In [10]:
df_se = df_se.drop(["Unnamed: 0"], axis = 1)

In [11]:
df_se

,UR,URFF,YUA,FDI,QOR,QOA,EIAP,PDPS,P65,P014,...,FEM,LFG,EPFRG,GGAG,DI,GE,CPI,RQ,VAA,EUPC
0,6.16,6.47,19.26,9.42,5.65,6.03,2.25,22.0,17.44,16.90,...,76.5,0.07,1920.24,-2007.0,9.73,2.02,93.0,1.57,1.51,3.554298e+06
1,6.24,6.57,20.21,8.05,5.63,6.02,2.12,23.0,17.65,16.73,...,76.7,0.06,1074.44,-1243.0,9.88,1.95,93.0,1.64,1.52,5.290971e+06
2,8.35,8.01,25.02,2.06,5.69,6.04,2.18,23.0,17.94,16.63,...,75.1,0.01,1618.18,-3953.0,9.73,2.04,92.0,1.65,1.55,1.950399e+07
3,8.61,8.52,24.76,0.13,5.71,6.20,2.10,23.0,18.28,16.60,...,74.4,0.03,1729.81,5519.0,9.50,1.96,92.0,1.66,1.54,1.868423e+07
4,7.80,7.76,22.78,1.22,5.66,6.25,1.99,23.0,18.64,16.66,...,76.0,0.06,1966.61,-3806.0,9.50,1.93,93.0,1.89,1.61,3.370849e+07
5,7.98,7.71,23.61,0.78,5.58,5.97,2.06,23.0,18.98,16.79,...,76.4,0.03,1723.00,-2826.0,9.73,1.91,88.0,1.90,1.69,1.155998e+07
6,8.05,7.89,23.52,0.25,5.52,5.73,2.03,24.0,19.27,16.97,...,76.7,0.06,1949.87,-1789.0,9.73,1.79,89.0,1.90,1.66,2.608015e+07
7,7.95,7.70,22.82,-1.05,5.50,5.67,1.97,24.0,19.52,17.16,...,77.2,0.04,650.56,-1307.0,9.73,1.68,87.0,1.81,1.61,3.553847e+07
8,7.43,7.29,20.36,1.95,5.36,5.60,2.04,24.0,19.70,17.34,...,77.9,0.05,5207.32,-20.0,9.45,1.68,89.0,1.80,1.57,8.000893e+06


In [12]:
se = pd.DataFrame()
se = df_se.copy()
se['GGFC_3'] = se['GGFC'].shift(3)
se['DI_3'] = se['DI'].shift(3)
se['EIAP_1'] = se['EIAP'].shift(1)

In [13]:
se = se.dropna()

In [14]:
sm = StructureModel()

In [15]:
sm.add_edges_from([
    ('EPFRG', 'EUPC'),
])

In [16]:
sm.edges

OutEdgeView([('EPFRG', 'EUPC')])

In [17]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_se_2007_2015.html")

Graphs/fully_connected_se_2007_2015.html


In [18]:
bn = BayesianNetwork(sm)

In [21]:
discretised_se = pd.DataFrame(index=se.index)

for col in se.columns:
    no_unique = se[col].nunique()
    
    if no_unique <= 1:
        discretised_se[col] = 0
    else:
        try:
            discretised_se[col] = pd.qcut(
                se[col], 
                q=min(3, no_unique),
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_se[col] = se[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_se)

Discretised data:
   UR  URFF  YUA  FDI  QOR  QOA  EIAP  PDPS  P65  P014  ...  GGAG  DI  GE  \
3   2     2    2    0    2    2     2     0    0     0  ...     2   0   2   
4   0     1    0    2    2    2     0     0    0     0  ...     0   0   2   
5   1     1    2    1    1    1     2     0    1     1  ...     0   1   1   
6   2     2    1    1    1    1     1     1    1     1  ...     1   1   1   
7   1     0    1    0    0    0     0     1    2     2  ...     1   1   0   
8   0     0    0    2    0    0     1     1    2     2  ...     2   0   0   

   CPI  RQ  VAA  EUPC  GGFC_3  DI_3  EIAP_1  
3    2   0    0     1       0     1       2  
4    2   1    1     2       0     2       2  
5    0   2    2     0       2     1       0  
6    1   2    2     1       1     0       1  
7    0   1    1     2       1     0       1  
8    1   0    0     0       2     1       0  

[6 rows x 35 columns]


In [22]:
discretised_se = discretised_se.reset_index(drop=True)
bn.fit_node_states(discretised_se)
baseline_auc = utils.get_avg_auc_all_info(discretised_se, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.2808899879455566 seconds
Processing fold 1 using 7 cores takes 2.970905065536499 seconds
Processing fold 2 using 7 cores takes 2.970923662185669 seconds
Processing fold 3 using 7 cores takes 2.9727816581726074 seconds
Processing fold 4 using 7 cores takes 2.8890328407287598 seconds
Baseline AUC: 0.38125


In [23]:
edges_to_add = [('LV', 'EPFRG'), ('LV', 'EUPC')]
edges_to_remove = [('EPFRG', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [24]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_se_2005_2015.html")

Graphs/node_added_se_2005_2015.html


In [25]:
discretised_se['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_se, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EPFRG' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.0839650630950928 seconds
Processing fold 1 using 7 cores takes 3.0838799476623535 seconds
Processing fold 2 using 7 cores takes 3.056190013885498 seconds
Processing fold 3 using 7 cores takes 3.045485734939575 seconds
Processing fold 4 using 7 cores takes 3.0617520809173584 seconds
AUC from adding LV between 'EPFRG' and 'EUPC': 0.675
